# MK-UNet on Colab

Phase 1 baseline for the tumor-segmentation project: trains MK-UNet on a polyp dataset
and reports Dice, IoU, HD95, sensitivity and specificity.

**Each session:** Runtime → Change runtime type → T4 GPU, then run sections 1–8 in order.
Colab destroys the machine between sessions, so the setup sections rerun every time.
Everything that must persist lives in Google Drive.

**First time only:** the dataset has to be in your Drive before section 6.
Run sections 1–3 (section 3 creates `My Drive/mkunet/data/`), then open the folder link
below, choose **Organize → Add shortcut to Drive**, and point it at `My Drive/mkunet/data/`.
A shortcut is a pointer, so it costs no quota.

- ClinicDB: <https://drive.google.com/drive/folders/1FPJr5f91uUCikxMvkwtZSEnYHemTZq1P>
- ColonDB: <https://drive.google.com/drive/folders/1u4_8dMztnEBUaX-w3XfUR3jXLBhpccPA>

## 1 · Configuration

The only cell you edit. Everything below reads from it.

In [ ]:
# Your fork
GITHUB_USER = "EliBaumgardner"
REPO        = "MK-UNET_Research"

# Experiment
DATASET   = "ClinicDB"    # ClinicDB | ColonDB
NETWORK   = "MK_UNet"     # MK_UNet_T | MK_UNet_S | MK_UNet | MK_UNet_M | MK_UNet_L
NUM_RUNS  = 1             # 1 through Phase 3; 5 at Phase 4 for mean +/- std
SEED      = 42            # run n uses SEED + n - 1
EPOCHS    = 200
BATCHSIZE = 8
IMG_SIZE  = 352
LR        = 0.0005

# Paths
DRIVE_ROOT = "/content/drive/MyDrive/mkunet"
REPO_DIR   = f"/content/{REPO}"
DATA_DST   = f"{REPO_DIR}/data/polyp/target"
PERSISTED  = ["model_pth", "logs", "results_polyp", "predictions_polyp"]

DATASET_FOLDER_IDS = {
    "ClinicDB": "1FPJr5f91uUCikxMvkwtZSEnYHemTZq1P",
    "ColonDB":  "1u4_8dMztnEBUaX-w3XfUR3jXLBhpccPA",
}

# Shared by the smoke test and the full run; --epoch is appended by each.
TRAIN_FLAGS = (f"--network {NETWORK} --dataset_name {DATASET} --num_runs {NUM_RUNS} "
               f"--seed {SEED} --batchsize {BATCHSIZE} --img_size {IMG_SIZE} --lr {LR}")

print(f"{NETWORK} on {DATASET} | {NUM_RUNS} run(s) x {EPOCHS} epochs | batch {BATCHSIZE} | seed {SEED}")

## 2 · Check the GPU

Fails loudly on a CPU runtime rather than training for hours at unusable speed.

In [ ]:
import subprocess

import torch

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print(gpu or "no GPU detected")
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU, then Connect."

## 3 · Mount Drive

Creates the `mkunet/` tree, including the `data/` folder the dataset shortcut goes into.

In [ ]:
import os

from google.colab import drive

drive.mount("/content/drive")
for sub in ["data"] + PERSISTED:
    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)

print("ready:", DRIVE_ROOT)

## 4 · Get the code

Clones your fork, or pulls if it is already there. This is the bridge from your Mac:
edit locally → `git push` → rerun this cell.

In [ ]:
import os
import subprocess


def sh(cmd, cwd=None):
    result = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    print((result.stdout + result.stderr).strip())
    return result.returncode


if os.path.isdir(f"{REPO_DIR}/.git"):
    sh("git pull", cwd=REPO_DIR)
else:
    sh(f"git clone https://github.com/{GITHUB_USER}/{REPO}.git {REPO_DIR}")

assert os.path.isdir(f"{REPO_DIR}/.git"), (
    f"Clone failed. Check that github.com/{GITHUB_USER}/{REPO} exists and is public.")

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

## 5 · Install dependencies

Six packages beyond Colab's defaults. `timm` is pinned because `mkunet_network.py:9-10`
imports from `timm.models.layers` and `timm.models.helpers`, shim paths that newer
releases have removed.

Ignore `requirements.txt`: it is inherited from EMCAD/CASCADE and most of what it lists
is never imported. See `ProjectFlow.md` for the audit.

In [ ]:
%pip install -q timm==0.9.16 albumentations medpy SimpleITK segmentation-mask-overlay thop openpyxl

## 6 · Stage the dataset

Copies the dataset from Drive to local disk, because training reads thousands of small
files and the Drive mount is slow for that. Fails with the paths it tried if the
shortcut step was skipped.

In [ ]:
import glob
import os
import subprocess

os.makedirs(DATA_DST, exist_ok=True)
dst = f"{DATA_DST}/{DATASET}"


def has_splits(path):
    return os.path.isdir(os.path.join(path, "train", "images"))


if has_splits(dst):
    print("already staged:", dst)
else:
    candidates = [
        f"{DRIVE_ROOT}/data/{DATASET}",
        f"/content/drive/MyDrive/{DATASET}",
        f"{DRIVE_ROOT}/data/target/{DATASET}",
        f"/content/drive/MyDrive/data/{DATASET}",
    ]
    src = next((c for c in candidates if has_splits(c)), None)
    if src is None:
        raise SystemExit(
            "Dataset not found in Drive. Checked:\n  " + "\n  ".join(candidates) +
            f"\n\nOpen https://drive.google.com/drive/folders/{DATASET_FOLDER_IDS[DATASET]}"
            "\nthen Organize -> Add shortcut to Drive -> My Drive/mkunet/data/ and rerun.")
    print(f"copying {src} -> {dst}")
    subprocess.run(f'cp -rL "{src}" "{dst}"', shell=True, check=True)

for split in ["train", "val", "test"]:
    images = len(glob.glob(f"{dst}/{split}/images/*"))
    masks = len(glob.glob(f"{dst}/{split}/masks/*"))
    print(f"  {split:<5} images {images:>4}  masks {masks:>4}")
    assert images > 0 and images == masks, f"{split}: {images} images but {masks} masks"

## 7 · Persist outputs to Drive

Checkpoints, logs and results are written inside the repo. These symlinks redirect them
into Drive so they survive the session ending.

In [ ]:
import os
import subprocess

for name in PERSISTED:
    local, remote = f"{REPO_DIR}/{name}", f"{DRIVE_ROOT}/{name}"
    os.makedirs(remote, exist_ok=True)
    if os.path.islink(local):
        continue
    if os.path.isdir(local):
        subprocess.run(f'rm -rf "{local}"', shell=True, check=True)
    os.symlink(remote, local)
    print(f"{name} -> {remote}")

# test_polyp.py appends the cross-run table to the repo root, which is not a symlinked
# directory, so the file itself is linked out.
summary = "All_Runs_Summary_Polyp.xlsx"
if not os.path.islink(f"{REPO_DIR}/{summary}"):
    os.symlink(f"{DRIVE_ROOT}/{summary}", f"{REPO_DIR}/{summary}")
    print(f"{summary} -> {DRIVE_ROOT}/{summary}")

## 8 · Sanity check

Resolves every import the training script needs, builds the model, and runs one forward
pass. Cheap insurance against finding a broken import forty minutes into training.

In [ ]:
import logging
import os

import torch

os.chdir(REPO_DIR)

from mkunet_network import MK_UNet
from utils.dataloader_polyp import get_loader
from utils.utils import cal_params_flops  # pulls in the long dependency chain
import openpyxl  # pandas needs it to write the results workbook

CHANNELS = {  # mirrors NET_CONFIGS in train_polyp.py
    "MK_UNet_T": [4, 8, 16, 24, 32],
    "MK_UNet_S": [8, 16, 32, 48, 80],
    "MK_UNet":   [16, 32, 64, 96, 160],
    "MK_UNet_M": [32, 64, 128, 192, 320],
    "MK_UNet_L": [64, 128, 256, 384, 512],
}

model = MK_UNet(num_classes=1, in_channels=3, channels=CHANNELS[NETWORK]).cuda()
out = model(torch.randn(2, 3, IMG_SIZE, IMG_SIZE).cuda())
out = out[0] if isinstance(out, (list, tuple)) else out

print(f"{NETWORK}: output {tuple(out.shape)}")
cal_params_flops(model, IMG_SIZE, logging.getLogger())

del model, out
torch.cuda.empty_cache()

## 9 · Smoke test

Two epochs, one run. Confirms the data loads, the loss falls and checkpoints get
written. The `Step [xxxx/yyyy]` counter gives batches per epoch, which is what you
multiply to estimate total runtime.

Its checkpoints land in `model_pth/` next to real runs, so note the `run_id` it prints —
section 11 picks the most recent directory.

In [ ]:
!cd {REPO_DIR} && python -W ignore train_polyp.py {TRAIN_FLAGS} --num_runs 1 --epoch 2

## 10 · Full training

Enable background execution first (Colab Pro) so the job survives closing the tab.

Each batch is trained at three scales (`train_polyp.py:112`) and both val and test are
scored every epoch, so an epoch costs roughly three times what the batch count suggests.
The best checkpoint is chosen on validation Dice, with the test score at that epoch
recorded separately — that is the correct protocol, not a bug.

In [ ]:
!cd {REPO_DIR} && python -W ignore train_polyp.py {TRAIN_FLAGS} --epoch {EPOCHS}

## 11 · Evaluate

Scores the best checkpoint on the test split and writes per-image metrics to
`results_polyp/`. `run_id` embeds a timestamp, so the newest run is found rather than
reconstructed.

In [ ]:
import glob
import os

runs = sorted(glob.glob(f"{REPO_DIR}/model_pth/*"), key=os.path.getmtime)
assert runs, "No runs found in model_pth/."

RUN_ID = os.path.basename(runs[-1])
print("evaluating:", RUN_ID)
if len(runs) > 1:
    print("other runs present:", *[os.path.basename(r) for r in runs[:-1]], sep="\n  ")

In [ ]:
!cd {REPO_DIR} && python -W ignore test_polyp.py --network {NETWORK} --run_id {RUN_ID} --dataset_name {DATASET}

## 12 · Results

The `AVERAGE` row is the Phase 1 baseline — record it in `ProjectFlow.md`; every later
change is measured against it.

HD95 caveat: `test_polyp.py:53` substitutes a fixed `100.0` for cases where the distance
is undefined (empty prediction) and averages that in rather than excluding it, so treat
HD95 as approximate. Dice and IoU are unaffected.

In [ ]:
import pandas as pd

results = pd.read_excel(f"{REPO_DIR}/results_polyp/Results_{RUN_ID}_{DATASET}_test.xlsx")
print(results.tail(1).to_string(index=False))
display(results)

---

## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| Assertion in section 2 | CPU runtime | Runtime → Change runtime type → T4 GPU |
| Clone assertion in section 4 | Fork name or visibility | Check `GITHUB_USER` and `REPO` in section 1 |
| Dataset assertion in section 6 | Shortcut not added | Follow the paths it prints; redo the Drive shortcut |
| Your code change had no effect | Stale clone | `git push` on the Mac, rerun section 4 |
| `ImportError` from `timm` | Wrong version | Rerun section 5; the 0.9.16 pin matters |
| Everything vanished | Session recycled | Expected. Rerun sections 1–8; Drive is intact |
| Training died partway | Session limit | There is no resume logic — the run restarts from scratch |

## Notes on this codebase

- `size_rates = [0.75, 1, 1.25]` (`train_polyp.py:112`) triples per-epoch cost.
- `--dataset_name`, `--num_runs` and `--seed` are additions to this fork; upstream
  hardcodes the dataset and five runs.
- `NET_CONFIGS` is duplicated across `train_polyp.py`, `test_polyp.py` and section 8.
  Changing channel widths in Phase 3 means changing all three.
- The `MK_UNet_T` and `MK_UNet_S` classes in `mkunet_network.py` are unused; every size
  is built from `MK_UNet` with different `channels`.